In [38]:
from langgraph.graph import MessagesState, StateGraph, END
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.types import Command, interrupt
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
from langchain_tavily import TavilySearch
from langchain.messages import AIMessage, HumanMessage,ToolMessage
from langgraph.checkpoint.memory import MemorySaver

In [39]:
memory=MemorySaver()

In [40]:
tavily_search=TavilySearch()
tools=[tavily_search]

In [41]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
llm_with_tools=llm.bind_tools(tools=tools)

In [42]:
prompt_template=ChatPromptTemplate([
    ("system","You are an AI Agent. You should not Hallucinate, If you do not know an answer use tools to find answers."),
    MessagesPlaceholder(variable_name="messages"),
    ("system","Do not use tool unnecessarely. Try to anwer user query using the previous messages."),
])

In [43]:
responder_chain=prompt_template | llm_with_tools

def responder_node(state:MessagesState)->Command:
    current_state=state
    result=responder_chain.invoke(current_state)
    if isinstance(result,AIMessage) and hasattr(result,"tool_calls") and len(result.tool_calls)>0:
        return Command(
            goto="tool_executer",
            update={"messages":[result]}
        )
    else:
        return Command(
            goto="feedback_receiver",
            update={"messages":[result]}
        )

def feedback_node(state:MessagesState)->dict:
    lastAIMessage=state["messages"][-1]
    feedback=interrupt(
        value={
            "Generated Post":lastAIMessage.content[0]["text"],
            # "Query":"Are you satisfied with the generated post?"
        }
    )
    if "done" in feedback.lower():
        return Command(
            goto=END
        )
    else:
        return Command(
            goto="responder",
            update={"messages":[HumanMessage(content=feedback)]}
        )

def tool_node(state:MessagesState)->dict:
    lastAIMessage=state["messages"][-1]
    tool_results=[]
    for tool_calls in lastAIMessage.tool_calls:
        tool_name=tool_calls["name"]
        tool_args=tool_calls["args"]
        tool_id=tool_calls["id"]
        tool_to_execute=next((tool for tool in tools if tool.name==tool_name),None)

        if tool_to_execute:
            tool_result=tool_to_execute.invoke(tool_args)
            tool_results.append(ToolMessage(content=tool_result,tool_call_id=tool_id ))
    
    return Command(
        goto="responder",
        update={"messages":tool_results}
    )

In [44]:
config={
    "configurable":{
        "thread_id":1
    }
}

graph=StateGraph(MessagesState)
graph.add_node("responder",responder_node)
graph.add_node("tool_executer",tool_node)
graph.add_node("feedback_receiver",feedback_node)
graph.set_entry_point("responder")

app=graph.compile(checkpointer=memory)

In [45]:
user_input=input("USER: ")
for chunk in app.stream({"messages":user_input}, config=config):
    print(chunk)

{'responder': {'messages': [AIMessage(content=[{'type': 'text', 'text': "Here's a draft for your LinkedIn post:\n\n---\n\n**Unlocking Hyper-Growth: How Startups Are Leveraging AI to Revolutionize Their Trajectories**\n\nThe startup landscape has always been synonymous with innovation, agility, and the relentless pursuit of growth. Today, a powerful catalyst is accelerating this journey like never before: Artificial Intelligence. AI is no longer a futuristic concept; it's a present-day imperative, and forward-thinking startups are harnessing its capabilities to not just compete, but to truly dominate their niches.\n\nFor many nascent companies, resources are finite, and efficiency is paramount. This is where AI truly shines. Imagine automating repetitive tasks that once consumed countless hours – from customer support chatbots handling routine inquiries to AI-powered tools streamlining data entry, content generation, or even HR processes. By offloading these operational burdens, startup

In [57]:
while True:
    snapshot=app.get_state(config=config)
    if snapshot.tasks and snapshot.tasks[0].interrupts:
        post = snapshot.tasks[0].interrupts[0].value['Generated Post']
        print(f"******GENERATED POST********\n{post}\nAre you satisfied with the generated post?")
        user_feedback=input("USER: ")
        app.invoke(Command(resume=user_feedback),config=config)
    else:
        break

******GENERATED POST********
Here's a draft for your LinkedIn post:

---

**Unlocking Hyper-Growth: How Startups Are Leveraging AI to Revolutionize Their Trajectories**

The startup landscape has always been synonymous with innovation, agility, and the relentless pursuit of growth. Today, a powerful catalyst is accelerating this journey like never before: Artificial Intelligence. AI is no longer a futuristic concept; it's a present-day imperative, and forward-thinking startups are harnessing its capabilities to not just compete, but to truly dominate their niches.

For many nascent companies, resources are finite, and efficiency is paramount. This is where AI truly shines. Imagine automating repetitive tasks that once consumed countless hours – from customer support chatbots handling routine inquiries to AI-powered tools streamlining data entry, content generation, or even HR processes. By offloading these operational burdens, startups free up their most valuable asset: human talent. E

In [71]:
snapshot=app.get_state(config=config)
# if snapshot.tasks and snapshot.tasks[0].interrupts:
print(snapshot.values["messages"][-1].content[0]["text"])


Alright, let's inject some humor into that LinkedIn post! Get ready for some startup shenanigans and AI wizardry.

---

**Warning: May Contain Excessive Enthusiasm & AI-Powered Shenanigans. Startups, Listen Up!**

Let's be real, the startup world is a glorious, chaotic mess of late-night coding, cold coffee, and the constant fear that your brilliant idea might just be a fever dream. We're all chasing that elusive "hyper-growth" like it's the last slice of pizza at a hackathon. And guess what's showing up with a cape and a super-powered algorithm to help? You guessed it: **Artificial Intelligence!**

Forget those pre-AI days when "manual process" was practically a four-letter word and your customer support was just you, frantically typing replies at 3 AM. AI isn't just a fancy buzzword; it's the intern who never sleeps, never complains, and actually *likes* spreadsheets.

Imagine this:
*   **Customer Service:** Your chatbot, "Sir-Bots-A-Lot," handles FAQs faster than you can say "server